# core

> `InteractiveShell` services that kernel clients drive remotely: introspection, evaluation, completion, signature help, and display publishing


In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import typing, warnings, traceback, tokenize, inspect
from ast import literal_eval
from io import StringIO
from collections.abc import Mapping
from types import ModuleType, FunctionType, MethodType, BuiltinFunctionType
from inspect import signature
from fastcore.utils import patch, dict2obj
from fastcore.aio import maybe_await
from fastcore.funccall import get_schema_nm
from jedi import Interpreter, Script as jscript
from IPython.core.interactiveshell import InteractiveShell
from IPython.core.completer import ProvisionalCompleterWarning
from IPython.core.display import DisplayObject


In [ ]:
from fastcore.test import *
from fastcore.utils import *
from pprint import pprint
from IPython.display import HTML


In [ ]:
warnings.filterwarnings('ignore', category=ProvisionalCompleterWarning)

## InteractiveShell helpers

In [ ]:
def _safe_repr(obj, max_len=200):
    "Safely get the repr() of an object, truncating if it exceeds max_len."
    try:
        s = str(obj)
        return s[:max_len] + ("…" if len(s)>max_len else "")
    except Exception as e: return f"<repr error: {str(e)}>"

In [ ]:
s = "Some long string that will be truncated"
print(_safe_repr(s, max_len=20))

Some long string tha…


In [ ]:
o = dict(name="Example", data=[1,2,3,4,5] * 5, nested=dict(a=1, b=2, c=[3, 4, 5] * 10))
print(_safe_repr(o, max_len=40))

{'name': 'Example', 'data': [1, 2, 3, 4,…


In [ ]:
def _safe_sig(v):
    try: return str(signature(v))
    except Exception: return '(...)'

@patch
def user_items(self:InteractiveShell, max_len=200, xtra_skip=()):
    "Get user-defined vars & funcs from namespace."
    ns,nsh = self.user_ns,self.user_ns_hidden
    ignore = {'nbmeta', 'receive_nbmeta'}
    ignore.add(xtra_skip)
    rm_types = (type, FunctionType, ModuleType, MethodType, BuiltinFunctionType,
        *(getattr(typing, o, ()) for o in ('_SpecialGenericAlias', '_GenericAlias', '_SpecialForm')))
    user_items = {k: v for k, v in ns.items() if not k in ignore and k not in nsh}
    user_vars = {k:_safe_repr(v, max_len=max_len)
        for k, v in user_items.items() if not k.startswith('_') and not isinstance(v, rm_types)}
    user_fns = {k:_safe_sig(v) for k, v in user_items.items()
        if isinstance(v, FunctionType) and v.__module__ == '__main__' and not k.startswith('__')}
    return user_vars,user_fns

In [ ]:
# ipy = get_ipython()
# _vs,_fs = ipy.user_items()
# pprint(_vs)
# print('---')
# pprint(_fs)

In [ ]:
def _rank(c, s):
    "Rank a completion `c` for text `s` with namespace `ns`."
    parts = s.split('.')
    is_public = not c.text.startswith('_')
    if c.type=='param': r=1
    elif c.mod=='__main__': r=2 # local
    elif len(parts)>1 and parts[0]==c.mod: r=3 # module
    elif c.mod=='builtins': r=4
    else: r=5
    return r if is_public else r+0.1

In [ ]:
@patch
def ranked_complete(self:InteractiveShell, code, line_no=None, col_no=None):
    ns = self.user_ns
    lines = code.splitlines(True)
    if line_no: offset = sum(len(lines[i]) for i in range(line_no-1)) + col_no -1
    else: offset = len(code)
    cs = self.Completer.completions(code, offset)
    def _c(a):
        res = dict2obj(text=a.text, type=str(a.type or ''), signature=str(getattr(a, 'signature', '') or ''), start=a.start, end=a.end)
        res['mod'] = getattr(ns.get(a.text, None), '__module__', None)
        res['rank'] = _rank(res, s=code)
        return res
    return [_c(c) for c in cs if not c.text.startswith('__') or '__' in code]

In [ ]:
from random import random

In [ ]:
ipy = get_ipython()

In [ ]:
def range_ex(
    a:str # some param
):
    "some func docstring"
    ...
ipy.ranked_complete('rang')

[{'text': 'range',
  'type': 'class',
  'signature': '',
  'start': 0,
  'end': 4,
  'mod': None,
  'rank': 5},
 {'text': 'range_ex',
  'type': 'function',
  'signature': '(a: str)',
  'start': 0,
  'end': 4,
  'mod': '__main__',
  'rank': 2},
 {'text': 'range_of',
  'type': 'function',
  'signature': '(a, b=None, step=None)',
  'start': 0,
  'end': 4,
  'mod': 'fastcore.basics',
  'rank': 5}]

In [ ]:
res = ipy.ranked_complete('a="foo"\na.', 2, 3)
res[:2]

[{'text': 'capitalize',
  'type': 'function',
  'signature': '() -> LiteralString',
  'start': 10,
  'end': 10,
  'mod': None,
  'rank': 5},
 {'text': 'casefold',
  'type': 'function',
  'signature': '() -> LiteralString',
  'start': 10,
  'end': 10,
  'mod': None,
  'rank': 5}]

In [ ]:
class Foo:
    def __dir__(self): return ['bar', 'baz', '_secret', 'quux']
    bar = 42
    baz = "hello"

f = Foo()
[o['text'] for o in ipy.ranked_complete('f.')]

['bar', 'baz', 'quux']

In [ ]:
def _maybe_eval(o):
    try:
        literal_eval(repr(o))
        return o
    except: return str(o)

In [ ]:
@patch
def get_vars(self:InteractiveShell, vs:list, literal=True):
    "Get variables from namespace."
    ns = self.user_ns
    return {v:_maybe_eval(ns[v]) if literal else str(ns[v]) for v in vs if v in ns}

In [ ]:
x, y, fp = 3, 4, open('./00_core.ipynb')
def add(a,b): return a+b

In [ ]:
test_eq(ipy.get_vars(['x', 'y', 'fp']).values(), [x,y,str(fp)])
test_eq(ipy.get_vars(['x', 'y', 'fp'], False).values(), [str(x),str(y),str(fp)])

In [ ]:
@patch
async def eval_exprs(self:InteractiveShell, vs:list, literal=True):
    "Evaluate expressions in namespace."
    ns,res = self.user_ns,{}
    for v in vs:
        try:
            e = await maybe_await(eval(v, ns))
            res[v] = _maybe_eval(e) if literal else str(e)
        except Exception as e: res[v] = f'<error type="{type(e).__name__}" desc="{e}">\n{traceback.format_exc()}</error>'
    return res

In [ ]:
print((await ipy.eval_exprs(['1/0']))['1/0'])

<error type="ZeroDivisionError" desc="division by zero">
Traceback (most recent call last):
  File "/var/folders/51/b2_szf2945n072c0vj2cyty40000gn/T/ipymini_57881/194151691.py", line 7, in eval_exprs
    e = await maybe_await(eval(v, ns))
                          ~~~~^^^^^^^
  File "<string>", line 1, in <module>
ZeroDivisionError: division by zero
</error>


In [ ]:
test_eq(await ipy.eval_exprs(['x', 'y']), {'x': 3, 'y': 4})
test_eq(await ipy.eval_exprs(['add(1,2)']), {'add(1,2)': 3})
test_eq(await ipy.eval_exprs(['x+y']), {'x+y': 7})
test_eq(await ipy.eval_exprs(['[x, y]']), {'[x, y]': [3, 4]})
test((await ipy.eval_exprs(['undefined_var']))['undefined_var'], "NameError: name 'undefined_var' is not defined", operator.contains)

In [ ]:
def _get_schema(ns: dict, t):
    "Check if tool `t` has errors."
    try: schema = get_schema_nm(t, ns, pname='parameters', evalable=True, skip_hidden=True, dot2dash=True)
    except (KeyError, AttributeError): return f"`{t}` not found. Did you run it?"
    except Exception as e: return f"`{t}`: {e}."
    return {'type':'function', 'function':schema}

@patch
def get_schemas(self:InteractiveShell, fs:list):
    "Get schemas from namespace."
    return {f:_get_schema(self.user_ns, f) for f in fs}

In [ ]:
ipy.get_schemas(['range_ex'])

{'range_ex': {'type': 'function',
  'function': {'name': 'range_ex',
   'description': 'some func docstring',
   'parameters': {'type': 'object',
    'properties': {'a': {'description': 'some param', 'type': 'string'}},
    'required': ['a']}}}}

Errors are passed back as strings:

In [ ]:
def add(a:int,b:int): return a + b
ipy.get_schemas(['add'])

{'add': '`add`: Docstring missing!.'}

In [ ]:
ipy.get_schemas(['div'])

{'div': '`div` not found. Did you run it?'}

Dotted names (like `obj.method`) are supported for getting schemas from object attributes:

In [ ]:
class Calculator:
    def add(self, a:int, b:int) -> int:
        "Add two numbers"
        return a + b

calc = Calculator()
ipy.get_schemas(['calc.add'])

{'calc.add': {'type': 'function',
  'function': {'name': 'calc-add',
   'description': 'Add two numbers\n\nReturns:\n- type: integer',
   'parameters': {'type': 'object',
    'properties': {'a': {'description': '', 'type': 'integer'},
     'b': {'description': '', 'type': 'integer'}},
    'required': ['a', 'b']}}}}

### Signatures

In [ ]:
def _signatures(ns, s, line, col):
    ctx = Interpreter(s, [ns]).get_signatures(line, col)
    if not ctx: ctx = jscript(s).get_signatures(line, col)
    return ctx

@patch
def _sig_jedi(self:InteractiveShell, code, line_no=None, col_no=None):
    ns = self.user_ns
    ctx = _signatures(ns, code, line=line_no, col=col_no)
    if ctx:
        def _s(s): return dict(label=s.description, typ=s.type, mod=s.module_name, doc=s.docstring(), idx=s.index,
            params=[dict(name=p.name, desc=p.description) for p in s.params])
        return [_s(opt) for opt in ctx]
    return []

In [ ]:
res = ipy._sig_jedi('truncstr("a",', 1, 13)
r = res[0]
test_eq(r['params'][r['idx']]['name'], 'maxlen')
res

[{'label': 'def truncstr',
  'typ': 'function',
  'mod': 'fastcore.xtras',
  'doc': "truncstr(s: str, maxlen: int, suf: str='…', space='', sizevar: str=None)\n\nTruncate `s` to length `maxlen`, adding suffix `suf` if truncated",
  'idx': 1,
  'params': [{'name': 's', 'desc': 'param s: str'},
   {'name': 'maxlen', 'desc': 'param maxlen: int'},
   {'name': 'suf', 'desc': "param suf: str='…'"},
   {'name': 'space', 'desc': "param space=''"},
   {'name': 'sizevar', 'desc': 'param sizevar: str=None'}]}]

In [ ]:
def _param_idx(code, cursor_pos):
    toks = []
    def op(s): return s == tokenize.OP
    g = tokenize.generate_tokens(StringIO(code[:cursor_pos]).readline)
    while True:
        try: toks.append(next(g))
        except (tokenize.TokenError, StopIteration): break
    depth,idx = 0,0
    for t in reversed(toks):
        if op(t.type) and t.string in ')]}': depth += 1
        elif op(t.type) and t.string in '([{':
            if depth == 0: return idx
            depth -= 1
        elif op(t.type) and t.string == ',' and depth == 0: idx += 1
    return 0

In [ ]:
_param_idx('print(1, 2, ', 12), _param_idx('print(foo(1,2), ', 16), _param_idx('print("a,b", ', 13)

(2, 1, 1)

In [ ]:
@patch
def _sig_dyn(self:InteractiveShell, code, line_no=None, col_no=None):
    from IPython.utils.tokenutil import token_at_cursor
    lines = code.splitlines()
    cursor_pos = sum(len(l)+1 for l in lines[:line_no-1])+col_no if line_no else len(code)
    name = token_at_cursor(code, cursor_pos)
    info = self._object_find(name)
    if not (info.found and callable(info.obj)): return []
    try: sig = inspect.signature(info.obj)
    except (ValueError, TypeError): return []
    ps = list(sig.parameters.values())
    idx = _param_idx(code, cursor_pos)
    vp = next((i for i, p in enumerate(ps) if p.kind is p.VAR_POSITIONAL), None)
    if vp is not None and idx > vp: idx = vp   # *args soaks the positional overflow
    return [dict(label=name, typ=type(info.obj).__name__, mod=getattr(info.obj,'__module__',''), idx=idx,
        doc=getattr(info.obj,'__doc__','') or '', params=[dict(name=p.name, desc=str(p)) for p in ps])]

In [ ]:
s = 'truncstr("a",'
res = ipy._sig_dyn(s, 1, len(s))
r = res[0]
test_eq(r['params'][r['idx']]['name'], 'maxlen')
res

[{'label': 'truncstr',
  'typ': 'function',
  'mod': 'fastcore.xtras',
  'doc': 'Truncate `s` to length `maxlen`, adding suffix `suf` if truncated',
  'idx': 1,
  'params': [{'name': 's', 'desc': 's: str'},
   {'name': 'maxlen', 'desc': 'maxlen: int'},
   {'name': 'suf', 'desc': "suf: str = '…'"},
   {'name': 'space', 'desc': "space=''"},
   {'name': 'sizevar', 'desc': 'sizevar: str = None'}]}]

In [ ]:
@patch
def sig_help(self:InteractiveShell, code, line_no=None, col_no=None):
    "Get signature help for code at cursor position using dynamic analysis or jedi as a backup."
    return self._sig_dyn(code, line_no, col_no=col_no) or self._sig_jedi(code, line_no, col_no=col_no)

In [ ]:
class _DynObj:
    def __getattr__(self, name):
        def _inner(x, y=1): ...
        _inner.__name__ = name
        return _inner

_dyn = _DynObj()

# Dynamic path: Jedi can't resolve __getattr__:
res = ipy.sig_help('_dyn.foo(', 1, 10)
test_eq(res[0]['label'], '_dyn.foo')
test_eq(len(res[0]['params']), 2)
res

[{'label': '_dyn.foo',
  'typ': 'function',
  'mod': '__main__',
  'doc': '',
  'idx': 0,
  'params': [{'name': 'x', 'desc': 'x'}, {'name': 'y', 'desc': 'y=1'}]}]

In [ ]:
test_eq(ipy.sig_help('print("a", ', line_no=1, col_no=11)[0]['idx'], 0)  # *args soaks the positional overflow

## Displaying MIME data

In [ ]:
cts = '#### A heading\n\nThis is **bold**.'
md_bundle = { 'text/markdown': cts }
ipy.display_pub.publish(data=md_bundle)

#### A heading

This is **bold**.

In [ ]:
@patch
def publish(self:InteractiveShell, data='', subtype='plain', mimetype='text', meta=None, update=False, **kw):
    if isinstance(data, DisplayObject): data,_ = self.display_formatter.format(data)
    elif not isinstance(data, Mapping): data = {f'{mimetype}/{subtype}': data}
    self.display_pub.publish(data, metadata=meta, transient=kw, update=update)

In [ ]:
ipy.publish(cts, 'markdown', foo='bar')

#### A heading

This is **bold**.

In [ ]:
ipy.publish(HTML('<b>hi</b> there'))

HTML(<b>hi</b> there)

In [ ]:
ipy.publish({'text/plain':'hi there'})

hi there

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()